# Hardware y Escala — Módulo 15

Este notebook es el compañero práctico del módulo de arquitectura. Aquí vas a:

1. Inspeccionar el hardware de tu propia máquina desde Python
2. Ver la diferencia de vectorización con tus propios ojos (y temporizador)
3. Observar efectos de caché en código real
4. Medir FLOPs empíricamente en multiplicaciones de matrices
5. Reproducir las gráficas del módulo y explorar los datos
6. Estimar costos de entrenamiento de LLMs interactivamente

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/15_arquitectura_de_computadoras/code/01_arquitectura.ipynb)

In [ ]:
%pip install numpy matplotlib psutil -q

In [ ]:
import platform
import psutil
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # para compatibilidad en Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print(f"Python:    {platform.python_version()}")
print(f"NumPy:     {np.__version__}")
print(f"Plataforma: {platform.system()} {platform.machine()}")

---
## Sección 1: Tu hardware

Python puede interrogar al sistema operativo sobre el hardware disponible.
Esto es útil para entender en qué entorno corre tu código y qué puedes esperar de él.

In [ ]:
# Información de CPU
print("=== CPU ===")
print(f"Arquitectura:       {platform.machine()}")
print(f"Procesador:         {platform.processor() or 'N/D (ver /proc/cpuinfo)'}")
print(f"Núcleos físicos:    {psutil.cpu_count(logical=False)}")
print(f"Núcleos lógicos:    {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Frecuencia actual:  {freq.current:.0f} MHz")
    print(f"Frecuencia máx:     {freq.max:.0f} MHz")

In [ ]:
# Información de memoria y almacenamiento
print("=== Memoria ===")
ram = psutil.virtual_memory()
print(f"RAM total:          {ram.total / 2**30:.1f} GB")
print(f"RAM disponible:     {ram.available / 2**30:.1f} GB")
print(f"RAM en uso:         {ram.percent:.1f}%")

print("\n=== Almacenamiento ===")
for part in psutil.disk_partitions()[:2]:
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"{part.mountpoint}: {usage.total / 2**30:.0f} GB total, "
              f"{usage.free / 2**30:.0f} GB libres")
    except PermissionError:
        pass

In [ ]:
# ¿Hay GPU disponible?
print("=== GPU ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"CUDA disponible: SÍ")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory
        print(f"VRAM: {mem / 2**30:.1f} GB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("Metal (Apple GPU): disponible")
    else:
        print("Sin GPU acelerada disponible — usando CPU")
except ImportError:
    print("PyTorch no instalado — no se puede verificar GPU")
    print("En Colab: Runtime > Change runtime type > GPU para activarla")

> **💡 Prueba esto:** Compara tus resultados con los de alguien más en la clase.
> ¿Cuántos núcleos físicos vs lógicos tiene cada máquina? ¿Cuánta RAM?
> En Colab (CPU runtime): ¿cuántos núcleos obtienes? ¿Cuánto RAM?

In [ ]:
cores_phys = psutil.cpu_count(logical=False)
cores_log = psutil.cpu_count(logical=True)
ram_gb = psutil.virtual_memory().total / 2**30
threads_per_core = (cores_log / cores_phys) if cores_phys else float("nan")

print("=== Resumen de esta máquina ===")
print(f"Núcleos físicos : {cores_phys}")
print(f"Núcleos lógicos : {cores_log}")
print(f"RAM total       : {ram_gb:.1f} GB")
print(f"Hilos por núcleo: {threads_per_core:.2f}x")

if threads_per_core >= 2:
    print("\nInterpretación: parece que hay SMT/Hyper-Threading (más hilos lógicos que físicos).")
else:
    print("\nInterpretación: físicos y lógicos casi coinciden; no se aprecia SMT fuerte.")

print("\n=== Plantilla para comparar con alguien más ===")
rows = [
    ("Esta máquina", cores_phys, cores_log, f"{ram_gb:.1f} GB"),
    ("Compañero/a", "_____ ", "_____ ", "_____ "),
    ("Colab CPU", "ejecuta este notebook en Colab", "", ""),
]
print(f"{'Equipo':<18} {'Físicos':>8} {'Lógicos':>8} {'RAM':>14}")
print("-" * 52)
for equipo, fis, log, ram in rows:
    print(f"{equipo:<18} {str(fis):>8} {str(log):>8} {str(ram):>14}")

print("\nNota: en Colab los recursos cambian entre sesiones; esta celda te deja la tabla lista para llenarla.")

---
## Sección 2: Vectorización — ver la diferencia

Esta es la demostración más importante del notebook.
Vamos a hacer exactamente la misma operación de tres formas y medir el tiempo.

In [ ]:
N = 10_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

In [ ]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")

In [ ]:
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"

In [ ]:
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")

> **💡 Prueba esto:**
> - Cambia `N` a 100_000 y a 100_000_000. ¿Cómo cambia el speedup?
> - Prueba con `np.float32` en vez de `np.float64`. ¿Es más rápido?
> - ¿Qué pasa si haces `a_np * b_np` (multiplicación) en vez de suma?

In [ ]:
def bytes_human(n):
    unidades = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    x = float(n)
    while x >= 1024 and i < len(unidades) - 1:
        x /= 1024
        i += 1
    return f"{x:.1f} {unidades[i]}"

def benchmark_python_estimate(N, op="add", sample_N=1_000_000):
    """
    Mide loop Python en un tamaño manejable y extrapola linealmente a N.
    """
    sample_N = min(N, sample_N)
    a_s = list(range(sample_N))
    b_s = list(range(sample_N, 2 * sample_N))

    t0 = time.perf_counter()
    if op == "add":
        _ = [a_s[i] + b_s[i] for i in range(sample_N)]
    elif op == "mul":
        _ = [a_s[i] * b_s[i] for i in range(sample_N)]
    else:
        raise ValueError("op debe ser 'add' o 'mul'")
    t = time.perf_counter() - t0

    return {
        "sample_N": sample_N,
        "sample_time": t,
        "estimated_time_full": t * (N / sample_N),
        "mode": "medido" if sample_N == N else "estimado"
    }

def benchmark_numpy_safe(N, dtype=np.float64, op="add", max_frac_ram=0.50):
    """
    Intenta medir NumPy para el N completo; si no hay RAM suficiente, mide un chunk
    y extrapola linealmente.
    """
    itemsize = np.dtype(dtype).itemsize
    est_bytes = 3 * N * itemsize  # a, b, resultado
    available = psutil.virtual_memory().available
    can_run_full = est_bytes < available * max_frac_ram

    if can_run_full:
        n_run = N
        mode = "medido"
    else:
        n_run = min(N, max(100_000, int((available * max_frac_ram) // (3 * itemsize))))
        mode = "estimado"

    a_np = np.arange(n_run, dtype=dtype)
    b_np = np.arange(n_run, 2 * n_run, dtype=dtype)

    t0 = time.perf_counter()
    if op == "add":
        _ = a_np + b_np
    elif op == "mul":
        _ = a_np * b_np
    else:
        raise ValueError("op debe ser 'add' o 'mul'")
    t = time.perf_counter() - t0

    t_full = t if n_run == N else t * (N / n_run)
    return {
        "time_full": t_full,
        "mode": mode,
        "bytes_est": est_bytes,
        "n_run": n_run,
    }

experimentos = [
    (100_000, np.float64, "add"),
    (10_000_000, np.float64, "add"),
    (100_000_000, np.float64, "add"),
    (10_000_000, np.float32, "add"),
    (10_000_000, np.float64, "mul"),
]

print(f"{'N':>12}  {'dtype':>8}  {'op':>4}  {'Loop Python':>16}  {'NumPy':>16}  {'Speedup':>10}  {'RAM est.':>10}")
print("-" * 100)

for N_exp, dtype_exp, op_exp in experimentos:
    py = benchmark_python_estimate(N_exp, op=op_exp)
    npy = benchmark_numpy_safe(N_exp, dtype=dtype_exp, op=op_exp)

    speedup = py["estimated_time_full"] / npy["time_full"] if npy["time_full"] > 0 else float("inf")
    py_label = f"{py['estimated_time_full']:.4f}s ({py['mode']})"
    np_label = f"{npy['time_full']:.4f}s ({npy['mode']})"

    print(f"{N_exp:>12,}  {np.dtype(dtype_exp).name:>8}  {op_exp:>4}  {py_label:>16}  {np_label:>16}  {speedup:>9.1f}×  {bytes_human(npy['bytes_est']):>10}")

print("\nLectura rápida:")
print("- Para N pequeño, el overhead fijo pesa más y el speedup suele ser menor.")
print("- Conforme N crece, NumPy amortiza mejor el overhead y el speedup normalmente aumenta.")
print("- float32 usa la mitad de memoria que float64; a veces también corre más rápido.")
print("- La multiplicación suele ser un poco más costosa que la suma, aunque ambas siguen siendo vectorizadas.")
print("\nNota: cuando aparece 'estimado', la celda evitó usar demasiada RAM y extrapoló desde un chunk más pequeño.")

---
## Sección 3: Efectos de caché

Acceder a datos en orden secuencial (cache-friendly) es dramáticamente más rápido
que acceder de forma aleatoria, aunque el número de operaciones sea idéntico.

Esto no es sobre Python — es sobre física del hardware.

In [ ]:
# Creamos una matriz cuadrada grande
SIZE = 2000
M = np.random.randn(SIZE, SIZE).astype(np.float32)
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")

In [ ]:
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")

In [ ]:
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

### ¿Por qué ocurre esto?

Cuando NumPy lee `M[i, :]` (una fila), los datos están contiguos en memoria —
el procesador los carga en caché en bloques de 64 bytes (cache lines).
Las siguientes lecturas ya están en caché.

Cuando lee `M[:, j]` (una columna), cada elemento está a `SIZE × 4 = 8,000 bytes`
del anterior. Cada acceso es un cache miss: hay que ir a buscar a RAM.

```
Fila [i, :]:   [0][1][2][3][4]...  ← contiguos, 1 cache miss por bloque
Columna [:, j]: [0]....[1]....[2].. ← separados, 1 cache miss por elemento
```

En pandas: `.iterrows()` accede por filas a un DataFrame que internamente
almacena columnas. Por eso es especialmente lento.

> **💡 Prueba esto:**
> - Cambia `SIZE` a 200. ¿Desaparece el efecto? (La matriz cabe en caché L3)
> - Prueba con `SIZE = 5000`. ¿Se amplifica el efecto?
> - ¿Qué pasa si conviertes la columna a array contiguo antes: `M[:, j].copy().sum()`?

In [ ]:
def benchmark_cache(size, dtype=np.float32):
    M_local = np.random.randn(size, size).astype(dtype)

    t0 = time.perf_counter()
    total_rows = 0.0
    for i in range(size):
        total_rows += M_local[i, :].sum()
    t_rows = time.perf_counter() - t0

    t0 = time.perf_counter()
    total_cols = 0.0
    for j in range(size):
        total_cols += M_local[:, j].sum()
    t_cols = time.perf_counter() - t0

    t0 = time.perf_counter()
    total_cols_copy = 0.0
    for j in range(size):
        total_cols_copy += M_local[:, j].copy().sum()
    t_cols_copy = time.perf_counter() - t0

    return {
        "size": size,
        "mb": M_local.nbytes / 1e6,
        "rows_s": t_rows,
        "cols_s": t_cols,
        "cols_copy_s": t_cols_copy,
        "ratio_cols_rows": t_cols / t_rows if t_rows else float("nan"),
        "ratio_copy_rows": t_cols_copy / t_rows if t_rows else float("nan"),
        "check": abs(total_rows - total_cols) + abs(total_rows - total_cols_copy),
    }

sizes_extra = [200, SIZE, 5000]
resultados_cache = []

print(f"{'SIZE':>6}  {'MB':>8}  {'filas (s)':>10}  {'columnas (s)':>13}  {'col.copy (s)':>13}  {'col/filas':>10}")
print("-" * 78)
for s in sizes_extra:
    r = benchmark_cache(s)
    resultados_cache.append(r)
    print(f"{r['size']:>6}  {r['mb']:>8.1f}  {r['rows_s']:>10.4f}  {r['cols_s']:>13.4f}  {r['cols_copy_s']:>13.4f}  {r['ratio_cols_rows']:>9.2f}×")

print("\nInterpretación esperada:")
print("- SIZE=200: la matriz es pequeña y suele caber mejor en caché; el efecto se reduce.")
print("- SIZE grande: el acceso por columnas castiga más porque los datos no están contiguos en memoria.")
print("- Hacer copy() de la columna fuerza un bloque contiguo temporal; a veces mejora respecto a la columna pura, aunque copiar también cuesta.")

---
## Sección 4: Multiplicación de matrices y FLOPs

La operación central de las redes neuronales. Vamos a medirla empíricamente
y calcular cuántos FLOPs realmente estamos ejecutando.

In [ ]:
def benchmark_matmul(N, reps=3):
    """Mide tiempo de N×N matmul y calcula GFLOPS observados."""
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)

    # Warm-up
    _ = A @ B

    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)

    t_med = np.median(times)
    flops = 2 * N**3           # FLOPs teóricos: 2×M×N×K con M=N=K
    gflops = flops / t_med / 1e9
    return t_med, gflops

sizes = [64, 128, 256, 512, 1024, 2048]
print(f"{'N':>6}  {'Tiempo (ms)':>12}  {'GFLOPS obs.':>12}")
print("-" * 36)
results = []
for N in sizes:
    t, g = benchmark_matmul(N)
    results.append((N, t, g))
    print(f"{N:>6}  {t*1000:>12.2f}  {g:>12.1f}")

In [ ]:
# Graficar GFLOPS observados vs N
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor="#1a1a2e")

for ax in [ax1, ax2]:
    ax.set_facecolor("#16213e")
    for sp in ax.spines.values():
        sp.set_edgecolor("#2a2a4e")
    ax.tick_params(colors="#e0e0e0")
    ax.xaxis.label.set_color("#e0e0e0")
    ax.yaxis.label.set_color("#e0e0e0")
    ax.title.set_color("#e0e0e0")
    ax.yaxis.grid(True, color="#2a2a4e", alpha=0.5)
    ax.set_axisbelow(True)

ns     = [r[0] for r in results]
times  = [r[1] * 1000 for r in results]  # ms
gflops = [r[2] for r in results]

ax1.plot(ns, times, 'o-', color="#0db7ed", linewidth=2, markersize=7)
ax1.set_xlabel("Tamaño de matriz N")
ax1.set_ylabel("Tiempo (ms)")
ax1.set_title("Tiempo de N×N matmul")

ax2.plot(ns, gflops, 's-', color="#f0a500", linewidth=2, markersize=7)
ax2.set_xlabel("Tamaño de matriz N")
ax2.set_ylabel("GFLOPS observados")
ax2.set_title("GFLOPS observados — crece con N")

fig.suptitle("Multiplicación de matrices: tiempo y GFLOPS",
             color="#e0e0e0", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("matmul_benchmark.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()
print("\n¿Por qué los GFLOPS crecen con N?")
print("Con matrices más grandes, la GPU (o BLAS) puede saturar sus núcleos.")
print("Matrices pequeñas tienen overhead de setup relativo mayor.")

> **💡 Prueba esto:**
> - Compara `np.float32` vs `np.float64`. ¿Cuánto cambia el rendimiento?
> - Si tienes GPU disponible: prueba con `torch.matmul` en CUDA y compara.
> - ¿Qué GFLOPS teóricos tiene tu CPU? (Busca el modelo y sus specs.)

In [ ]:
def benchmark_matmul_dtype(N=1024, dtype=np.float32, reps=3):
    A = np.random.randn(N, N).astype(dtype)
    B = np.random.randn(N, N).astype(dtype)
    _ = A @ B  # warm-up

    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        _ = A @ B
        tiempos.append(time.perf_counter() - t0)

    t_med = float(np.median(tiempos))
    flops = 2 * N**3
    gflops = flops / t_med / 1e9
    return t_med, gflops

N_cmp = 1024
t32, g32 = benchmark_matmul_dtype(N_cmp, np.float32)
t64, g64 = benchmark_matmul_dtype(N_cmp, np.float64)

print("=== Matmul CPU por dtype ===")
print(f"float32 -> {t32*1000:.2f} ms, {g32:.2f} GFLOPS")
print(f"float64 -> {t64*1000:.2f} ms, {g64:.2f} GFLOPS")
print(f"Relación float32/float64 en GFLOPS: {g32/g64:.2f}×")

# Si hay GPU, comparar también con PyTorch CUDA
print("\n=== Intento de benchmark en GPU (PyTorch) ===")
try:
    import torch
    if torch.cuda.is_available():
        device = "cuda"
        A = torch.randn((N_cmp, N_cmp), device=device, dtype=torch.float32)
        B = torch.randn((N_cmp, N_cmp), device=device, dtype=torch.float32)

        _ = A @ B
        torch.cuda.synchronize()

        tiempos_gpu = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = A @ B
            torch.cuda.synchronize()
            tiempos_gpu.append(time.perf_counter() - t0)

        t_gpu = float(np.median(tiempos_gpu))
        g_gpu = (2 * N_cmp**3) / t_gpu / 1e9
        print("CUDA disponible: sí")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"torch.matmul float32 -> {t_gpu*1000:.2f} ms, {g_gpu:.2f} GFLOPS")
    else:
        print("No hay GPU CUDA disponible en este entorno.")
except Exception as e:
    print(f"No se pudo correr la parte de GPU: {e}")

print("\n=== Estimación teórica de GFLOPS de la CPU ===")
cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
freq_info = psutil.cpu_freq()
freq_ghz = ((freq_info.max or freq_info.current) / 1000.0) if freq_info else 2.5

flags_text = ""
try:
    with open("/proc/cpuinfo", "r", encoding="utf-8", errors="ignore") as f:
        flags_text = f.read().lower()
except Exception:
    pass

if "avx512f" in flags_text:
    fp32_per_cycle = 64
    fp64_per_cycle = 32
    vector_hint = "AVX-512 + FMA (heurístico)"
elif "avx2" in flags_text and "fma" in flags_text:
    fp32_per_cycle = 32
    fp64_per_cycle = 16
    vector_hint = "AVX2 + FMA (heurístico)"
elif "avx" in flags_text:
    fp32_per_cycle = 16
    fp64_per_cycle = 8
    vector_hint = "AVX (heurístico)"
else:
    fp32_per_cycle = 8
    fp64_per_cycle = 4
    vector_hint = "sin AVX claro / estimación conservadora"

theoretical_fp32 = cores * freq_ghz * fp32_per_cycle
theoretical_fp64 = cores * freq_ghz * fp64_per_cycle

print(f"Supuesto vectorial: {vector_hint}")
print(f"Núcleos físicos   : {cores}")
print(f"Frecuencia usada  : {freq_ghz:.2f} GHz")
print(f"Techo teórico FP32: ~{theoretical_fp32:.1f} GFLOPS")
print(f"Techo teórico FP64: ~{theoretical_fp64:.1f} GFLOPS")
print("\nNota: esto es una cota superior muy aproximada; el benchmark real suele quedar por debajo por memoria, caché, BLAS, tamaño del problema y overhead.")

---
## Sección 5: Reproduce el gráfico histórico de FLOPs

Los datos del módulo, reproducibles y modificables.

In [ ]:
# Datos históricos: (año, nombre, USD por GFLOP)
# Fuentes: diversos estudios de Karl Rupp, Our World in Data, benchmarks públicos
flops_history = [
    (1961, "IBM 7090",           1e10),
    (1984, "Cray X-MP",          4.2e7),
    (1994, "Intel Pentium",      3e4),
    (1997, "Intel Pentium II",   1e3),
    (2001, "AMD Athlon XP",      1e2),
    (2006, "PlayStation 3",      1.0),
    (2008, "AMD Radeon HD 4870", 0.065),
    (2013, "NVIDIA GTX 780",     0.002),
    (2017, "NVIDIA GTX 1080",    3.5e-4),
    (2020, "NVIDIA RTX 3090",    4e-5),
    (2022, "NVIDIA RTX 4090",    1.5e-5),
    (2023, "NVIDIA H100",        2e-6),
]

years = [d[0] for d in flops_history]
costs = [d[2] for d in flops_history]
names = [d[1] for d in flops_history]

reduction = flops_history[0][2] / flops_history[-1][2]
print(f"Reducción total (1961 → 2023): {reduction:,.0f}×")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

# Colorear: CPU = azul, GPU = violeta, consola = verde
colors = [
    "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed",  # CPUs
    "#2ecc71",                                                # PS3
    "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0",  # GPUs
]

ax.plot(years, costs, color="#444466", linewidth=1, alpha=0.5, zorder=1)
for i, (yr, cost, name) in enumerate(zip(years, costs, names)):
    ax.scatter(yr, cost, color=colors[i], s=80, zorder=3,
               edgecolors="white", linewidths=0.5)
    ax.annotate(name, (yr, cost),
                xytext=(5, 3), textcoords="offset points",
                color="#e0e0e0", fontsize=8)

ax.set_yscale("log")
ax.set_xlabel("Año")
ax.set_ylabel("USD por GFLOP (escala log)")
ax.set_title(f"Costo histórico de 1 GFLOP — {reduction:,.0f}× más barato en 62 años",
             fontweight="bold")

import matplotlib.patches as mpatches
leg = [
    mpatches.Patch(color="#0db7ed", label="CPU"),
    mpatches.Patch(color="#892ca0", label="GPU"),
    mpatches.Patch(color="#2ecc71", label="Consola"),
]
ax.legend(handles=leg, facecolor="#16213e", labelcolor="#e0e0e0", edgecolor="#2a2a4e")

plt.tight_layout()
plt.savefig("flops_historico_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

> **💡 Prueba esto:**
> - Agrega chips nuevos que encuentres online (MI300X, RTX 5090, etc.)
> - Calcula el precio del chip actual y divídelo entre los TFLOPS para obtener USD/GFLOP
> - ¿Qué costo por GFLOP tiene tu propia computadora?

In [ ]:
chips_extra = [
    {"year": 2024, "name": "AMD MI300X (aprox.)",      "price_usd": 15000, "tflops_fp32": 163.4},
    {"year": 2025, "name": "NVIDIA RTX 5090 (aprox.)", "price_usd": 1999,  "tflops_fp32": 104.8},
    {"year": 2024, "name": "NVIDIA B200 (aprox.)",     "price_usd": 30000, "tflops_fp32": 180.0},
]

for chip in chips_extra:
    chip["usd_per_gflop"] = chip["price_usd"] / (chip["tflops_fp32"] * 1000)

print(f"{'Chip':<30} {'Precio USD':>12} {'TFLOPS FP32':>14} {'USD/GFLOP':>12}")
print("-" * 76)
for chip in chips_extra:
    print(f"{chip['name']:<30} {chip['price_usd']:>12,.0f} {chip['tflops_fp32']:>14.1f} {chip['usd_per_gflop']:>12.6f}")

precio_pc_usd = 1200  # <-- cambia este valor por el precio real de tu laptop/PC
gflops_pc = results[-1][2]
usd_per_gflop_pc = precio_pc_usd / gflops_pc if gflops_pc else float("inf")

print("\n=== Tu computadora (estimación usando el benchmark observado) ===")
print(f"Precio asumido del equipo: ${precio_pc_usd:,.0f} USD")
print(f"Rendimiento observado    : {gflops_pc:.2f} GFLOPS")
print(f"Costo por GFLOP          : ${usd_per_gflop_pc:.4f} USD/GFLOP")

fig, ax = plt.subplots(figsize=(12, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

ax.plot(years, costs, color="#4cc9f0", linewidth=2, marker="o", markersize=5, label="Histórico")
for x, y, name in flops_history:
    ax.annotate(name, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8, color="#cde7ff")

for chip in chips_extra:
    ax.scatter(chip["year"], chip["usd_per_gflop"], s=70, zorder=4)
    ax.annotate(chip["name"], (chip["year"], chip["usd_per_gflop"]),
                xytext=(5, -12), textcoords="offset points", fontsize=8, color="#ffd6a5")

ax.scatter(2026, usd_per_gflop_pc, s=80, marker="D", zorder=5)
ax.annotate("Tu compu (estimada)", (2026, usd_per_gflop_pc),
            xytext=(5, 5), textcoords="offset points", fontsize=8, color="#ffffff")

ax.set_yscale("log")
ax.set_xlabel("Año")
ax.set_ylabel("USD por GFLOP")
ax.set_title("Costo por GFLOP: histórico + chips recientes", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTip: cambia `precio_pc_usd` por el precio real de tu equipo para obtener tu propio costo por GFLOP.")

---
## Sección 6: Explorador de costos de LLMs

Una función para estimar cuánto costaría entrenar un modelo dado.

In [ ]:
def estimar_entrenamiento(
    params_B: float,          # parámetros del modelo en billones
    tokens_B: float,          # tokens de entrenamiento en billones
    gpu_tflops_bf16: float,   # rendimiento efectivo del GPU en TFLOPS BF16
    gpu_price_per_hour: float = 3.0,  # USD por hora por GPU
    gpu_efficiency: float = 0.35,     # eficiencia real (35% típico en clúster)
) -> dict:
    """
    Estima FLOPs, tiempo y costo de entrenamiento usando la fórmula de Chinchilla:
    FLOPs ≈ 6 × parámetros × tokens
    """
    params = params_B * 1e9
    tokens = tokens_B * 1e9

    flops_total = 6 * params * tokens

    # FLOPS/s efectivos de 1 GPU
    flops_per_sec_gpu = gpu_tflops_bf16 * 1e12 * gpu_efficiency

    tiempo_segundos = flops_total / flops_per_sec_gpu
    tiempo_horas = tiempo_segundos / 3600
    costo_usd = tiempo_horas * gpu_price_per_hour

    return {
        "flops_total": flops_total,
        "flops_total_str": f"{flops_total:.2e}",
        "gpu_horas": tiempo_horas,
        "costo_1_gpu_usd": costo_usd,
        "gpus_para_1_semana": tiempo_horas / (24 * 7),
    }

print("Función lista. Probemos con algunos ejemplos...")

In [ ]:
# H100 con ~1,000 TFLOPS BF16 (rendimiento pico)
H100_TFLOPS = 1000
H100_PRICE  = 3.0   # USD/hora (cloud spot)

ejemplos = [
    ("LLaMA 2 7B",   7,    1_400),   # 7B params, ~1.4T tokens
    ("LLaMA 2 70B",  70,   2_000),   # 70B params, 2T tokens
    ("GPT-3 175B",   175,  300),     # 175B params, 300B tokens
    ("Modelo 1B (tuyo)", 1, 20),     # 1B params, 20B tokens — un proyecto pequeño
]

print(f"{'Modelo':<22} {'FLOPs':>10}  {'GPU-horas':>12}  {'Costo 1 GPU':>14}  {'GPUs p/semana':>14}")
print("-" * 78)

for nombre, params_B, tokens_B in ejemplos:
    r = estimar_entrenamiento(params_B, tokens_B, H100_TFLOPS, H100_PRICE)
    print(f"{nombre:<22} {r['flops_total_str']:>10}  "
          f"{r['gpu_horas']:>12,.0f}  "
          f"${r['costo_1_gpu_usd']:>12,.0f}  "
          f"{r['gpus_para_1_semana']:>14,.0f}")

In [ ]:
# Visualizar: parámetros vs costo estimado
modelos_viz = [
    {"name": "BERT-Large",   "params_B": 0.34,  "flops": 1.5e20, "cost_M": 0.007},
    {"name": "GPT-2",        "params_B": 1.5,   "flops": 5e19,   "cost_M": 0.05},
    {"name": "T5-11B",       "params_B": 11,    "flops": 5e21,   "cost_M": 0.5},
    {"name": "GPT-3 175B",   "params_B": 175,   "flops": 3.1e23, "cost_M": 5},
    {"name": "PaLM 540B",    "params_B": 540,   "flops": 2.5e24, "cost_M": 50},
    {"name": "LLaMA 2 70B",  "params_B": 70,    "flops": 2e24,   "cost_M": 20},
    {"name": "GPT-4 (est.)", "params_B": 1800,  "flops": 2e25,   "cost_M": 100},
]

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.xaxis.grid(True, color="#2a2a4e", alpha=0.2)
ax.set_axisbelow(True)

colors_m = plt.cm.plasma(np.linspace(0.1, 0.9, len(modelos_viz)))
for i, m in enumerate(modelos_viz):
    ax.scatter(m["params_B"], m["cost_M"], s=100, color=colors_m[i],
               zorder=3, edgecolors="white", linewidths=0.5)
    ax.annotate(m["name"], (m["params_B"], m["cost_M"]),
                xytext=(6, 4), textcoords="offset points",
                color="#e0e0e0", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Parámetros (billones)")
ax.set_ylabel("Costo de entrenamiento (millones USD)")
ax.set_title("Parámetros vs Costo de entrenamiento", fontweight="bold")

plt.tight_layout()
plt.savefig("llm_costo_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

> **💡 Prueba esto:**
> - Llama a `estimar_entrenamiento` con tu propio modelo hipotético.
> - ¿Cuántas GPUs H100 necesitarías para terminar en 1 semana un modelo de 13B con 1T tokens?
> - ¿Qué pasa si cambias la eficiencia de 35% a 50%? ¿Qué hace que no sea 100%?

In [ ]:
import math

modelo_hip = estimar_entrenamiento(
    params_B=13,
    tokens_B=1000,            # 1T tokens
    gpu_tflops_bf16=H100_TFLOPS,
    gpu_price_per_hour=H100_PRICE,
    gpu_efficiency=0.35,
)

print("=== Modelo hipotético: 13B parámetros, 1T tokens ===")
print(f"FLOPs totales     : {modelo_hip['flops_total_str']}")
print(f"GPU-horas (1 GPU) : {modelo_hip['gpu_horas']:,.0f}")
print(f"Costo (1 GPU)     : ${modelo_hip['costo_1_gpu_usd']:,.0f}")
print(f"GPUs para 1 semana: {math.ceil(modelo_hip['gpus_para_1_semana'])}")

r35 = estimar_entrenamiento(13, 1000, H100_TFLOPS, H100_PRICE, gpu_efficiency=0.35)
r50 = estimar_entrenamiento(13, 1000, H100_TFLOPS, H100_PRICE, gpu_efficiency=0.50)

print("\n=== Efecto de subir eficiencia de 35% a 50% ===")
print(f"35% -> GPU-horas: {r35['gpu_horas']:,.0f}, GPUs/semana: {math.ceil(r35['gpus_para_1_semana'])}")
print(f"50% -> GPU-horas: {r50['gpu_horas']:,.0f}, GPUs/semana: {math.ceil(r50['gpus_para_1_semana'])}")
print(f"Reducción relativa de tiempo/costo: {(1 - r50['gpu_horas']/r35['gpu_horas'])*100:.1f}%")

print("\n¿Por qué la eficiencia no es 100%?")
motivos = [
    "comunicación entre GPUs y sincronización",
    "tiempo en cargar datos y batches",
    "operaciones no dominadas por matmul (optimizer, logs, checkpoints)",
    "desbalance entre workers / stragglers",
    "límites de memoria, bandwidth y fragmentación",
]
for m in motivos:
    print(f"- {m}")

---
## Ejercicio integrador

Tienes acceso a un clúster con 8 GPUs H100 (1,000 TFLOPS BF16 cada una, rendimiento
efectivo 35%, $3/hora cada una). Quieres entrenar un modelo de lenguaje.

In [ ]:
import math
import statistics
import time
import numpy as np

# Parámetros del ejercicio
n_gpus = 8
h100_tflops = 1000          # TFLOPS BF16 por GPU
eficiencia = 0.35           # rendimiento efectivo
precio_gpu_hora = 3.0       # USD/hora/GPU
horas_semana = 7 * 24
segundos_semana = horas_semana * 3600

# -----------------------------
# 1) Máximo tamaño de modelo en 1 semana
# -----------------------------
# Chinchilla: tokens = 20 * params
# FLOPs de entrenamiento: 6 * params * tokens = 120 * params^2
flops_disponibles_semana = n_gpus * h100_tflops * 1e12 * eficiencia * segundos_semana
params_max = math.sqrt(flops_disponibles_semana / 120)
tokens_max = 20 * params_max

print("1) Entrenamiento máximo en 1 semana con 8 H100")
print(f"   FLOPs disponibles: {flops_disponibles_semana:.3e}")
print(f"   Parámetros máximos: {params_max/1e9:.3f} B")
print(f"   Tokens de entrenamiento (Chinchilla): {tokens_max/1e9:.3f} B")

# -----------------------------
# 2) Costo total de ese entrenamiento
# -----------------------------
costo_total_semana = n_gpus * precio_gpu_hora * horas_semana
print("\n2) Costo total de 1 semana")
print(f"   Costo total: ${costo_total_semana:,.2f} USD")

# -----------------------------
# 3) Tamaño de modelo con presupuesto de $10,000
# -----------------------------
presupuesto = 10_000
horas_disponibles_presupuesto = presupuesto / (n_gpus * precio_gpu_hora)
segundos_disponibles_presupuesto = horas_disponibles_presupuesto * 3600
flops_disponibles_presupuesto = n_gpus * h100_tflops * 1e12 * eficiencia * segundos_disponibles_presupuesto
params_presupuesto = math.sqrt(flops_disponibles_presupuesto / 120)
tokens_presupuesto = 20 * params_presupuesto

print("\n3) Tamaño máximo con presupuesto de $10,000 USD")
print(f"   Horas de clúster disponibles: {horas_disponibles_presupuesto:.2f} h")
print(f"   Días equivalentes: {horas_disponibles_presupuesto/24:.2f} días")
print(f"   Parámetros máximos: {params_presupuesto/1e9:.3f} B")
print(f"   Tokens de entrenamiento (Chinchilla): {tokens_presupuesto/1e9:.3f} B")

# -----------------------------
# 4) Benchmark matmul 1024x1024 en esta máquina
# -----------------------------
n = 1024
A = np.random.rand(n, n).astype(np.float32)
B = np.random.rand(n, n).astype(np.float32)

_ = A @ B  # warmup

repeticiones = 5
tiempos = []
for _ in range(repeticiones):
    t0 = time.perf_counter()
    _ = A @ B
    tiempos.append(time.perf_counter() - t0)

flops_matmul = 2 * n**3
mejor_tiempo = min(tiempos)
tiempo_mediano = statistics.median(tiempos)
gflops_mejor = flops_matmul / mejor_tiempo / 1e9
gflops_mediano = flops_matmul / tiempo_mediano / 1e9

print("\n4) Benchmark de matmul 1024x1024")
print(f"   Tiempos medidos (s): {[round(t, 4) for t in tiempos]}")
print(f"   Mejor tiempo: {mejor_tiempo:.4f} s")
print(f"   Tiempo mediano: {tiempo_mediano:.4f} s")
print(f"   Rendimiento (mejor caso): {gflops_mejor:,.2f} GFLOPS")
print(f"   Rendimiento (mediana): {gflops_mediano:,.2f} GFLOPS")

h100_peak_flops_1s = h100_tflops * 1e12
h100_effective_flops_1s = h100_tflops * 1e12 * eficiencia

cpu_seconds_for_h100_peak = h100_peak_flops_1s / (gflops_mediano * 1e9)
cpu_seconds_for_h100_effective = h100_effective_flops_1s / (gflops_mediano * 1e9)

print("\n   ¿Cuánto tardaría esta CPU en hacer lo que 1 H100 hace en 1 segundo?")
print(f"   Usando pico teórico H100 (1000 TFLOPS): {cpu_seconds_for_h100_peak:,.0f} s  (~{cpu_seconds_for_h100_peak/3600:.2f} h)")
print(f"   Usando rendimiento efectivo 35%:        {cpu_seconds_for_h100_effective:,.0f} s  (~{cpu_seconds_for_h100_effective/3600:.2f} h)")

print("\nResumen rápido:")
print(f"- En 1 semana con 8 H100: ~{params_max/1e9:.2f}B parámetros y ~{tokens_max/1e9:.2f}B tokens")
print(f"- Costo total de esa semana: ${costo_total_semana:,.0f} USD")
print(f"- Con $10k: ~{params_presupuesto/1e9:.2f}B parámetros y ~{tokens_presupuesto/1e9:.2f}B tokens")
print(f"- Tu CPU en este benchmark logra ~{gflops_mediano:.2f} GFLOPS (mediana)")

<details>
<summary>Pista para el punto 1</summary>

```python
n_gpus = 8
h100_tflops = 1000
eficiencia = 0.35
semanas = 1

flops_disponibles = n_gpus * h100_tflops * 1e12 * eficiencia * semanas * 7 * 24 * 3600

# FLOPs = 6 × params × 20 × params = 120 × params²
# params = sqrt(FLOPs / 120)
import math
params = math.sqrt(flops_disponibles / 120)
print(f"Parámetros máximos: {params/1e9:.1f}B")
```
</details>

---
## Resumen

| Experimento | Lección |
|-------------|----------|
| Hardware inspection | Cada máquina tiene restricciones concretas: cores, RAM, ausencia de GPU |
| Vectorización | NumPy no es solo comodidad — usa instrucciones SIMD del hardware |
| Efectos de caché | El orden de acceso a memoria importa tanto como el número de operaciones |
| MatMul FLOPs | Matrices grandes saturan el hardware; pequeñas son overhead-bound |
| FLOPs histórico | El costo de cómputo cayó 5 billones de veces en 62 años |
| Escala LLMs | GPT-4 costó ~$100M en compute. LLaMA 2 7B: estimado en ~$3M. Fine-tuning: miles. |

**El mensaje central:** el hardware no es un detalle de implementación.
Es el presupuesto físico dentro del cual vive cada algoritmo.